In [1]:
!pip install -q pypdf
!pip install -q transformers einops accelerate langchain langchain_experimental bitsandbytes
!pip install -q sentence_transformers
!pip install -q llama_index llama-index
!pip install -q llama-index llama-index-readers-file pypdf
!pip install -U langchain langchain-community sentence-transformers
!pip install -Uq llama-index-llms-huggingface
!pip install -U langchain-huggingface sentence-transformers
!pip install bitsandbytes accelerate
!pip install llama-index-embeddings-langchain
!pip install -U unstructured unstructured[pdf] pathlib
!apt-get update --fix-missing
!apt-get install -y poppler-utils
!pip install qdrant-client llama-index-vector-stores-qdrant

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.13).
0 upgraded, 0 newly install

In [2]:
from pathlib import Path

from llama_index.readers.file import UnstructuredReader
from llama_index.core.schema import TextNode
from llama_index.core import VectorStoreIndex, StorageContext, Settings

from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.embeddings.langchain import LangchainEmbedding

from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings

from qdrant_client import QdrantClient

reader = UnstructuredReader()

files_path = Path('/content/data')

documents = []

for file in files_path.rglob("*.pdf"):
  docs = reader.load_data(file=str(file))
  for d in docs:
    d.metadata["file_name"] = file.name
  documents.extend(docs)

print(f"Loaded {len(documents)} raw document parts")


/tmp/ipykernel_11888/3531540625.py:10: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


Loaded 5 raw document parts


In [3]:
hf_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

embed_model = LangchainEmbedding(hf_embeddings)


/tmp/ipykernel_11888/2573348723.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  hf_embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
splitter = SemanticChunker(
    hf_embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=85
)

all_chunks = []

for doc in documents:
    chunks = splitter.create_documents([doc.text])

    for c in chunks:
        c.metadata = dict(doc.metadata)

    all_chunks.extend(chunks)

print(f"Total semantic chunks: {len(all_chunks)}")

Total semantic chunks: 123


In [5]:
for i, c in enumerate(all_chunks[:3]):
    print("\n" + "="*60)
    print(c.metadata)
    print(c.page_content[:300])


{'coordinates': '{"points": [[160.1367, 71.87250000000006], [160.1367, 115.72180000000003], [439.91670000000016, 115.72180000000003], [439.91670000000016, 71.87250000000006]], "system": "PixelSpace", "layout_width": 595.276, "layout_height": 841.89}', 'file_directory': '/content/data', 'filename': 'Application-Form-BusinessQuickStart.pdf', 'last_modified': '2026-06-12T11:58:38', 'page_number': 1, 'languages': '["eng"]', 'filetype': 'application/pdf', 'file_name': 'Application-Form-BusinessQuickStart.pdf'}
application form Business Quick Start - Premium

Welcome to Etisalat. Please complete this form if you are applying for Business Quick Start - Premium. Please note that incomplete information may cause delays in providing the service.

{'coordinates': '{"points": [[160.1367, 71.87250000000006], [160.1367, 115.72180000000003], [439.91670000000016, 115.72180000000003], [439.91670000000016, 71.87250000000006]], "system": "PixelSpace", "layout_width": 595.276, "layout_height": 841.89}', 

In [6]:
from llama_index.core.schema import TextNode

nodes = [
    TextNode(
        text=chunk.page_content,
        metadata={
            **chunk.metadata,
            "chunk_idx": i
        }
    )
    for i, chunk in enumerate(all_chunks)
]

print(f"Converted {len(nodes)} chunks to TextNodes")




Converted 123 chunks to TextNodes


In [7]:
client = QdrantClient(":memory:")  # change to path="..." or url="..." for production

vector_store = QdrantVectorStore(
    client=client,
    collection_name="pdf_chunks"
)

storage_context = StorageContext.from_defaults(
    vector_store=vector_store
)

In [8]:
import torch
from langchain_community.embeddings import HuggingFaceEmbeddings
from llama_index.core.prompts import PromptTemplate
from llama_index.llms.huggingface import HuggingFaceLLM
# System prompt — instructions given to the model before every question
system_prompt = """
You are a Q&A assistant.

Rules:
- Answer ONLY using the provided context.
- Be concise (max 3-5 sentences).
- DO NOT repeat the answer.
- STOP immediately after giving the final answer.
- Do not add phrases like "The answer is:".
"""

QA_PROMPT = PromptTemplate(
"""You are a precise assistant.

Use ONLY the context below.
Do NOT repeat answers.
Do NOT include "Answer:" labels.
Write only one final response.

Context:
{context_str}

Question:
{query_str}

Final answer:"""
)

llm = HuggingFaceLLM(
    context_window=4096,
    max_new_tokens=150,
    generate_kwargs={
        "temperature": 0.0,
        "do_sample": False,
        "repetition_penalty": 1.2,   # 🔥 IMPORTANT FIX
        "top_p": 1.0
    },
    tokenizer_name="Qwen/Qwen2.5-1.5B-Instruct",
    model_name="Qwen/Qwen2.5-1.5B-Instruct",
    device_map="auto"
)

print("LLM loaded successfully!")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

LLM loaded successfully!


In [9]:
Settings.llm = llm
Settings.embed_model = embed_model

In [10]:
index = VectorStoreIndex(
    nodes,
    storage_context=storage_context,
    embed_model=embed_model
)

print("Index built successfully in Qdrant")


/usr/local/lib/python3.12/dist-packages/llama_index/vector_stores/qdrant/base.py:852: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  self._client.create_payload_index(


Index built successfully in Qdrant


In [11]:
query_engine = index.as_query_engine(
    text_qa_template=QA_PROMPT,
    similarity_top_k=2,
    response_mode="compact"
)


print("Query engine ready")

Query engine ready


In [12]:
def ask(question):
    print(f"\n❓ {question}")
    response = query_engine.query(question)
    print(f"\n🤖 {response}")
    print("\n📚 Sources:")
    for node in response.source_nodes:
        print(f"  score={node.score:.3f} | {node.text[:150]}...")



In [13]:
print(ask("What exactly is LOA and what is needed for application?"))


❓ What exactly is LOA and what is needed for application?


The following generation flags are not valid and may be ignored: ['temperature', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



🤖  A Letter Of Authorization (LOA), also known as an authorization letter, is typically used when someone wants to authorize another person's actions on their behalf within certain limits. In your document, it states that copies of all trade licenses need to be provided along with at least one authorized employee's name and details being mandatory. Additionally, you're asked about declaring authority over specific departments/entities based on your own capacities and authorizations. Lastly, there's mention of liability protection against potential legal issues related to service failures caused by unauthorized representatives' lapses. For these requirements, please refer back to the original text which outlines detailed instructions regarding providing necessary documentation like official-use-only checks, valid IDs, trade licenses, etc., while ensuring proper signatures and stamps accompany each

📚 Sources:
  score=0.267 | Copy of all trade licenses needs to be provided. at least one